In [8]:
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('database', exist_ok=True)

print("✅ Created necessary folders!")
import pandas as pd
import numpy as np
import re

print("="*60)
print("🧹 DATA CLEANING PROCESS")
print("="*60)

# Load raw data
print("\n1️⃣ Loading raw data...")
df = pd.read_csv(r'C:\Users\yelle\Job_Market_Analytics\data\raw\future_jobs_dataset.csv')
print(f"   ✅ Loaded {len(df)} jobs")
print(f"   📐 Shape: {df.shape}")

# Check for missing values
print("\n2️⃣ Checking for missing values...")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "   ✅ No missing values!")

# Remove duplicates
print("\n3️⃣ Removing duplicates...")
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"   ✅ Removed {before - after} duplicates")

# Clean salary data
print("\n4️⃣ Cleaning salary data...")
# Remove jobs with invalid salaries
df = df[df['salary_min'] > 0]
df = df[df['salary_max'] > df['salary_min']]
df['salary_avg'] = (df['salary_min'] + df['salary_max']) / 2
print(f"   ✅ Salary range: ₹{df['salary_min'].min():,} - ₹{df['salary_max'].max():,}")
print(f"   💰 Average salary: ₹{df['salary_avg'].mean():,.0f}")

# Clean job titles
print("\n5️⃣ Standardizing job titles...")
df['job_title_clean'] = df['job_title'].str.title().str.strip()

# Clean location
print("\n6️⃣ Standardizing locations...")
df['location_clean'] = df['location'].str.strip().str.title()

# Convert posted_date to datetime
print("\n7️⃣ Converting date format...")
df['posted_date'] = pd.to_datetime(df['posted_date'])
df['month'] = df['posted_date'].dt.to_period('M')
df['day_of_week'] = df['posted_date'].dt.day_name()
print(f"   ✅ Date range: {df['posted_date'].min()} to {df['posted_date'].max()}")

# Extract individual skills (create separate columns)
print("\n8️⃣ Extracting skills...")
all_skills = ['Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Machine Learning',
              'TensorFlow', 'PyTorch', 'AWS', 'Azure', 'Google Cloud', 'R', 
              'JavaScript', 'Docker', 'Kubernetes', 'Airflow', 'Spark', 'Hadoop', 
              'DAX', 'Visualization', 'Git', 'Jupyter', 'Pandas', 'NumPy']

# Create binary columns for each skill
for skill in all_skills:
    df[skill] = df['required_skills'].str.contains(skill, case=False, na=False).astype(int)

print(f"   ✅ Created {len(all_skills)} skill columns")

# Count total skills per job
df['total_skills'] = df[all_skills].sum(axis=1)
print(f"   📊 Avg skills per job: {df['total_skills'].mean():.1f}")

# Filter to only Data Analyst related jobs (for focused analysis)
print("\n9️⃣ Filtering for Data Analyst roles...")
data_analyst_keywords = ['data analyst', 'business analyst', 'product analyst', 'research analyst', 'bi developer']
df_filtered = df[df['job_title_clean'].str.lower().isin(data_analyst_keywords) | 
                 df['required_skills'].str.contains('Python|SQL|Power BI', case=False)]

print(f"   ✅ Filtered dataset: {len(df_filtered)} jobs")

# Save cleaned data
print("\n🔟 Saving cleaned data...")
df.to_csv('data/cleaned/job_postings_cleaned.csv', index=False)
df_filtered.to_csv('data/cleaned/data_analyst_jobs.csv', index=False)

print("="*60)
print("✅ DATA CLEANING COMPLETE!")
print("="*60)
print(f"\n📁 Files saved:")
print(f"   - data/cleaned/job_postings_cleaned.csv (all jobs)")
print(f"   - data/cleaned/data_analyst_jobs.csv (analyst roles only)")

# Display cleaned data
print("\n📊 Preview of cleaned data:")
display(df_filtered[['job_title_clean', 'company', 'location_clean', 'salary_avg', 
                     'job_type', 'experience_level', 'required_skills']].head(10))

✅ Created necessary folders!
🧹 DATA CLEANING PROCESS

1️⃣ Loading raw data...
   ✅ Loaded 500 jobs
   📐 Shape: (500, 13)

2️⃣ Checking for missing values...
   ✅ No missing values!

3️⃣ Removing duplicates...
   ✅ Removed 0 duplicates

4️⃣ Cleaning salary data...
   ✅ Salary range: ₹302,396 - ₹1,999,751
   💰 Average salary: ₹985,382

5️⃣ Standardizing job titles...

6️⃣ Standardizing locations...

7️⃣ Converting date format...
   ✅ Date range: 2026-03-07 00:00:00 to 2026-06-03 00:00:00

8️⃣ Extracting skills...
   ✅ Created 24 skill columns
   📊 Avg skills per job: 5.3

9️⃣ Filtering for Data Analyst roles...
   ✅ Filtered dataset: 376 jobs

🔟 Saving cleaned data...
✅ DATA CLEANING COMPLETE!

📁 Files saved:
   - data/cleaned/job_postings_cleaned.csv (all jobs)
   - data/cleaned/data_analyst_jobs.csv (analyst roles only)

📊 Preview of cleaned data:


,job_title_clean,company,location_clean,salary_avg,job_type,experience_level,required_skills
0,Research Analyst,DataFirst,Hyderabad,1168904.5,Remote,Mid,"Excel, Tableau, Azure, TensorFlow, Machine Lea..."
1,Ml Engineer,DataFirst,Mumbai,1100918.5,Hybrid,Mid,"Docker, Azure, SQL, Power BI, DAX"
2,Senior Data Analyst,DataFirst,Mumbai,1215264.0,Onsite,Senior,"Spark, Python, TensorFlow, Kubernetes"
3,Analytics Manager,StartupXYZ,Bangalore,818722.5,Remote,Mid,"Docker, Power BI, JavaScript, Visualization, T..."
4,Data Engineer,DataFlow,Chennai,895228.0,Remote,Senior,"Excel, AWS, Python, Kubernetes, DAX, SQL"
5,Research Analyst,Digital Solutions,Hyderabad,1039404.5,Remote,Lead,"Visualization, TensorFlow, Power BI, Docker, A..."
6,Data Analyst Junior,CloudSystems,Bangalore,1252135.5,Onsite,Entry,"Kubernetes, SQL, AWS, DAX, Hadoop, Tableau"
7,Business Analyst,CloudSystems,Bangalore,864002.0,Remote,Entry,"Kubernetes, Azure, Visualization, Tableau, Jav..."
8,Research Analyst,DataFlow,Hyderabad,1294847.0,Onsite,Mid,"Tableau, Docker, Python"
12,Data Engineer,DataFirst,Hyderabad,1118492.0,Remote,Senior,"SQL, DAX, Spark, Tableau, Excel"
